# Setup

In [1]:
# define root path of project 
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

# load general setup
from utils.setup_general import *

# load GIS setup
from utils.setup_gis_python import *

# Hacer queries a servidores
import requests 

Setup general cargado
Setup GIS cargado


# Problema general
- La ANM publica la base de datos de títulos mineros identificando el municipio correspondiente y el mineral que se explota https://www.datos.gov.co/Minas-y-Energ-a/ANM-RUCOM-Explotador-Minero-Autorizado-T-tulo-Mine/42ha-fhvj/about_data
- El problema es que no publica directamente el área de cada título minero ni los polígonos
- Sin embargo, si se conoce el número del expediente del título minero, los polígonos se pueden descargar del visor de la ANM https://annamineria.anm.gov.co/Html5Viewer/index.html?viewer=SIGMExt&locale=es-CO&appAcronym=sigm
- El video de esta página es una guía de la ANM de cómo hacerlo https://www.anm.gov.co/ventanilla-minera
- El problema es que descargarlos manualmente es demasiado extenso y suceptible a errores porque es un alto número de polígonos. Sólo para el oro son alrededor de 270 polígonos.
- Encontré una forma de importar los políginos con un punto de acceso al servidor de arcgis. 
- La idea es replicar queries como este que devulven una respuesta en json https://annamineria.anm.gov.co/annageo/rest/services/SIGM/TenureLayers/MapServer/4/query?f=json&where=LOWER(CODIGO_EXPEDIENTE)%20LIKE%20%27%25abq-101%25%27&returnGeometry=true&spatialRel=esriSpatialRelIntersects&outFields=*&outSR=102100https://annamineria.anm.gov.co/annageo/rest/services/SIGM/TenureLayers/MapServer/4/query?f=json&where=LOWER(CODIGO_EXPEDIENTE)%20LIKE%20%27%25abq-101%25%27&returnGeometry=true&spatialRel=esriSpatialRelIntersects&outFields=*&outSR=102100
- Esa respuesta después se puede convertir en un geodataframe
- En este cuaderno hago el query respectivo para cada titulo minero de oro en la base de títulos mineros de la ANM

# Ejemplo básico

In [2]:
# ruta de acceso
url = "https://annamineria.anm.gov.co/annageo/rest/services/SIGM/TenureLayers/MapServer/4/query"

# Parametros de la consulta
params = {
    "f": "json",
    "where": "CODIGO_EXPEDIENTE = 'ABQ-101'",
    "returnGeometry": "true",
    "outFields": "*",
    "outSR": "102100"
}

# Consultar
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

# Explorar resultado
print(response.url)
print(response.status_code)
print(response.headers.get("content-type"))
print(response.text[:500])

# Convertir a json
data = response.json()
# Convertir ArcGIS JSON a GeoJSON
geojson_dict = arcgis2geojson(data)

# Pasar a GeoDataFrame
gdf = gpd.GeoDataFrame.from_features(geojson_dict["features"])
# Asignar CRS con el que vienen los datos
gdf = gdf.set_crs(epsg=102100, allow_override=True)

Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}


https://annamineria.anm.gov.co/annageo/rest/services/SIGM/TenureLayers/MapServer/4/query?f=json&where=CODIGO_EXPEDIENTE+%3D+%27ABQ-101%27&returnGeometry=true&outFields=%2A&outSR=102100
200
application/json;charset=UTF-8
{"displayFieldName":"CODIGO_EXPEDIENTE","fieldAliases":{"CODIGO_EXPEDIENTE":"CODIGO_EXPEDIENTE","MODALIDAD":"MODALIDAD","TITULO_ESTADO":"TITULO_ESTADO","AREA_HA":"AREA_HA","CLASIFICACION_MINERIA":"CLASIFICACION_MINERIA","ETAPA":"ETAPA","SOLICITANTES_O_TITULARES":"SOLICITANTES_O_TITULARES","MINERALES":"MINERALES","FECHA_DE_SOLICITUD":"FECHA_DE_SOLICITUD","FECHA_DE_EXPEDICION":"FECHA_DE_EXPEDICION","FECHA_DE_ANIVERSARIO":"FECHA_DE_ANIVERSARIO","FECHA_DE_EXPIRACION":"FECHA_DE_EXPIRACION","PUBLICADO


In [3]:
gdf.head()

,geometry,CODIGO_EXPEDIENTE,MODALIDAD,TITULO_ESTADO,AREA_HA,CLASIFICACION_MINERIA,ETAPA,SOLICITANTES_O_TITULARES,MINERALES,FECHA_DE_SOLICITUD,FECHA_DE_EXPEDICION,FECHA_DE_ANIVERSARIO,FECHA_DE_EXPIRACION,PUBLICADO_EN_RUCOM,PAR,TITLE_TYPE_CODE,TENURE_STATUS_CODE,MINING_CLASSIFICATION_CODE,TENURE_STAGE_CODE,CENTROID_COORDINATE,TENURE_ID,OBJECTID,MUNICIPIOS,DEPARTAMENTOS,MINERALES_INACTIVOS,TIPO_TERMINACION,TERMINATION_TYPE_CODE,ACTIVE_TENURE_STATUS_IND,ACTIVE_APPLICATION_STATUS_IND
0,"POLYGON ((-8517471.673 397325.576, -8517471.29...",ABQ-101,CONTRATO DE CONCESIÓN (D 2655),Activo,39.08,Mediana,Explotación,(61017) CANTERA LA EMILIA SAS,"DIABASA, RECEBO",920023200000,1015286400000,1015286400000,2002060800000,Y,PAR CALI,CCD2,A,MED,EXPT,"-76.51667,3.56964",ABQ-101,176246,YUMBO,Valle del Cauca,None,None,None,Y,N


# Cargar lista de todos los titulos mineros

In [2]:
# Cargar información de títutlos mineros
titulos_mineros = pd.read_excel(
    io = DATA/'raw/aac_ANM_titulos_mineros/ANM_RUCOM_Explotador_Minero_Autorizado-Título_Minero_20260702.xlsx'
)

In [3]:
# Extraer únicamente titulos mineros de minerales de oro
titulos_mineros_oro = titulos_mineros[titulos_mineros['MINERAL'].str.contains('ORO')]

# Extraer identificadores de los expedientes de titulos mineros
expedientes_titulosMineros = list(titulos_mineros['CODIGO_EXPEDIENTE'].unique())

# Extraer identificadores de los expedientes de titulos oro
expedientes_oro = list(titulos_mineros_oro['CODIGO_EXPEDIENTE'].unique())

# Mostrar conteo de títulos mineros totales y de oro
print(f'Expendientes Totales: {len(expedientes_titulosMineros)}')
print(f'Expendientes de Oro: {len(expedientes_oro)}')

Expendientes Totales: 2807
Expendientes de Oro: 268


# Obtener poligonos de todos los expedientes

In [17]:
# ruta de acceso
url = "https://annamineria.anm.gov.co/annageo/rest/services/SIGM/TenureLayers/MapServer/4/query"

# Lista para almacenar poligonos
gdfs = []

# Para cada expediente
for exp in tqdm(expedientes_titulosMineros):
    
    # Definir parametros de query
    params = {
        "f": "json",
        "where": f"CODIGO_EXPEDIENTE = '{exp}'",
        "returnGeometry": "true",
        "outFields": "*",
        "outSR": "102100" # En la versión del query original sale con 102100, pero esa no es una forma estandar de especficarlo
    }
    
    # Obtener respuesta
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    
    if data.get("features"): # Si la respuesta existe
        geojson_dict = arcgis2geojson(data)
        gdf_temp = gpd.GeoDataFrame.from_features(geojson_dict["features"])
        gdf_temp = gdf_temp.set_crs(epsg=102100, allow_override=True)
        gdfs.append(gdf_temp)

 11%|█████████                                                                      | 321/2807 [00:54<06:44,  6.15it/s]Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}
Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}
 19%|███████████████▎                                                               | 545/2807 [01:31<06:02,  6.23it/s]Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}
Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}
 76%|███████████████████████████████████████████████████████████▏                  | 2129/2807 [05:47<01:53,  5.98it/s]Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}
Object converted in non-standard crs - {'wkid': 102100, 'latestWkid': 3857}
 77%|████████████████████████████████████████████████████████████                  | 2160/2807 [05:52<01:41,  6.36it/s]Object converted in non-standard crs - {'wkid': 102100, 'latestWkid'

ReadTimeout: HTTPSConnectionPool(host='annamineria.anm.gov.co', port=443): Read timed out. (read timeout=30)

In [16]:
# El servidor rompió la conexion cuando llevaba 2567 titulos mineros
# Por eso tuve que volver a correr el código para los faltantes

In [10]:
# ruta de acceso
url = "https://annamineria.anm.gov.co/annageo/rest/services/SIGM/TenureLayers/MapServer/4/query"

# Lista para almacenar poligonos
gdfs_p2 = []

# Para cada expediente
for exp in tqdm(expedientes_titulosMineros[2566-1:-1]):
    
    # Definir parametros de query
    params = {
        "f": "json",
        "where": f"CODIGO_EXPEDIENTE = '{exp}'",
        "returnGeometry": "true",
        "outFields": "*",
        "outSR": "102100" # En la versión del query original sale con 102100, pero esa no es una forma estandar de especficarlo
    }
    
    # Obtener respuesta
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    
    if data.get("features"): # Si la respuesta existe
        geojson_dict = arcgis2geojson(data)
        gdf_temp = gpd.GeoDataFrame.from_features(geojson_dict["features"])
        gdf_temp = gdf_temp.set_crs(epsg=102100, allow_override=True)
        gdfs_p2.append(gdf_temp)

100%|████████████████████████████████████████████████████████████████████████████████| 241/241 [00:40<00:00,  5.92it/s]


In [13]:
# Organizar resultados
gdf_final = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs="EPSG:102100")
gdf_final_p2 = gpd.GeoDataFrame(pd.concat(gdfs_p2, ignore_index=True), crs="EPSG:102100")

In [30]:
# Juntar las dos partes del df
gdf_poligonos_titulos_mineros = pd.concat([gdf_final, gdf_final_p2])

In [29]:
# mostrar informacion de los gdfs
display(gdf_final.info())
display(gdf_final_p2.info())
display(gdf_poligonos_titulos_mineros.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2535 entries, 0 to 2534
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   geometry                       2535 non-null   object 
 1   CODIGO_EXPEDIENTE              2535 non-null   object 
 2   MODALIDAD                      2535 non-null   object 
 3   TITULO_ESTADO                  2535 non-null   object 
 4   AREA_HA                        2535 non-null   float64
 5   CLASIFICACION_MINERIA          2507 non-null   object 
 6   ETAPA                          2535 non-null   object 
 7   SOLICITANTES_O_TITULARES       2535 non-null   object 
 8   MINERALES                      2535 non-null   object 
 9   FECHA_DE_SOLICITUD             2535 non-null   int64  
 10  FECHA_DE_EXPEDICION            2535 non-null   int64  
 11  FECHA_DE_ANIVERSARIO           2535 non-null   int64  
 12  FECHA_DE_EXPIRACION            2479 non-null   f

None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   geometry                       237 non-null    object 
 1   CODIGO_EXPEDIENTE              237 non-null    object 
 2   MODALIDAD                      237 non-null    object 
 3   TITULO_ESTADO                  237 non-null    object 
 4   AREA_HA                        237 non-null    float64
 5   CLASIFICACION_MINERIA          237 non-null    object 
 6   ETAPA                          237 non-null    object 
 7   SOLICITANTES_O_TITULARES       237 non-null    object 
 8   MINERALES                      237 non-null    object 
 9   FECHA_DE_SOLICITUD             237 non-null    int64  
 10  FECHA_DE_EXPEDICION            237 non-null    int64  
 11  FECHA_DE_ANIVERSARIO           237 non-null    int64  
 12  FECHA_DE_EXPIRACION            231 non-null    flo

None

<class 'pandas.core.frame.DataFrame'>
Index: 2772 entries, 0 to 236
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   geometry                       2772 non-null   object 
 1   CODIGO_EXPEDIENTE              2772 non-null   object 
 2   MODALIDAD                      2772 non-null   object 
 3   TITULO_ESTADO                  2772 non-null   object 
 4   AREA_HA                        2772 non-null   float64
 5   CLASIFICACION_MINERIA          2744 non-null   object 
 6   ETAPA                          2772 non-null   object 
 7   SOLICITANTES_O_TITULARES       2772 non-null   object 
 8   MINERALES                      2772 non-null   object 
 9   FECHA_DE_SOLICITUD             2772 non-null   int64  
 10  FECHA_DE_EXPEDICION            2772 non-null   int64  
 11  FECHA_DE_ANIVERSARIO           2772 non-null   int64  
 12  FECHA_DE_EXPIRACION            2710 non-null   object 

None

In [31]:
# Exportar resultado
gdf_poligonos_titulos_mineros.to_parquet(DATA/"raw/aad_ANM_titulos_mineros_poligonos/e2001_poligonos_titulosmineros.parquet")